# Module 03 — Creating Snapshots

This module walks through the full `PUT /_snapshot/{repo}/{snapshot}` API,
exercising every option with real examples.

## Snapshot creation parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `indices` | string / list | `*` (all) | Indices / patterns to include |
| `include_global_state` | bool | `false` | Capture cluster state |
| `feature_states` | list | `[]` | Elastic feature data to include |
| `partial` | bool | `false` | Allow snapshot with unavailable shards |
| `ignore_unavailable` | bool | `false` | Skip missing indices instead of failing |
| `metadata` | object | `{}` | Arbitrary user metadata |
| `wait_for_completion` | bool | `false` | Block until done (query param) |

In [1]:
import time
from helpers import *

es = get_client()
wait_for_green(es)
register_fs_repo(es)

✓ Cluster health: yellow

✓ Repository 'training-fs-repo' registered at /usr/share/elasticsearch/snapshots

---
## 1. Minimal Snapshot (all indices, no global state)

In [2]:
heading('1. Minimal snapshot')

delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-minimal')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='snap-minimal',
    body={},  # all defaults: all open indices, no global state, no feature states
    wait_for_completion=True,
)

snap = es.snapshot.get(repository=FS_REPO_NAME, snapshot='snap-minimal')['snapshots'][0]
console.print(f"  Indices : {snap['indices']}")
console.print(f"  State   : {snap['state']}")
console.print(f"  Global  : {snap['include_global_state']}")

─────────────────────────────────────────────── 1. Minimal snapshot ───────────────────────────────────────────────

Indices : ['.internal.alerts-dataset.quality.alerts-default-000001', '.kibana_search_solution_9.3.0_001', 
'.internal.alerts-security.alerts-default-000001', '.ds-ilm-history-7-2026.04.06-000001', '.ml-annotations-000001',
'.apm-custom-link', '.apm-source-map', '.slo-observability.summary-v3.6', 
'.internal.alerts-observability.uptime.alerts-default-000001', '.kibana_usage_counters_9.3.0_001', 
'.kibana-siem-rule-migrations-prebuiltrules', '.ml-inference-000005', '.kibana_analytics_9.3.0_001', 
'.ds-kibana_sample_data_logs-2026.04.06-000001', '.internal.alerts-stack.alerts-default-000001', 
'.internal.alerts-ml.anomaly-detection-health.alerts-default-000001', '.ml-notifications-000002', 
'.internal.alerts-security.attack.discovery.alerts-default-000001', 
'.ds-.edr-workflow-insights-default-2026.04.06-000001', '.ml-inference-native-000002', 
'kibana_sample_data_ecommerce', '.inference', '.apm-agent-configuration', '.kibana_9.3.0_001', 
'.internal.alerts-ml.anomaly-detection.alerts-default-000001', '.security-profile-8', '.kibana_locks-000001', 
'.security-7', '.internal.alerts-observability.apm.alerts-default-000001', 
'.internal.alerts-observability.slo.alerts-default-000001', '.ds-.kibana-event-log-ds-2026.04.06-000001', 
'.kibana_alerting_cases_9.3.0_001', '.secrets-inference', 'kibana_sample_data_flights', 
'.internal.alerts-observability.metrics.alerts-default-000001', '.slo-observability.sli-v3.6', 
'.ds-.logs-elasticsearch.deprecation-default-2026.04.06-000001', 
'.internal.alerts-transform.health.alerts-default-000001', '.kibana_ingest_9.3.0_001', 
'.kibana_security_solution_9.3.0_001', '.slo-observability.summary-v3.6.temp', '.kibana_security_session_1', 
'.kibana_task_manager_9.3.0_001', '.integration_knowledge-7', '.internal.alerts-streams.alerts-default-000001', 
'.internal.alerts-observability.logs.alerts-default-000001', '.kibana-siem-rule-migrations-integrations', 
'.internal.alerts-default.alerts-default-000001', '.internal.alerts-observability.threshold.alerts-default-000001']

State   : SUCCESS

Global  : True

---
## 2. Selective Index Snapshots

Use comma-separated names, wildcards, and exclusions (`-`) to control which indices are included.

In [3]:
heading('2a. Single index')
delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-single-index')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='snap-single-index',
    body={'indices': ['kibana_sample_data_ecommerce'], 'include_global_state': False},
    wait_for_completion=True,
)
snap = es.snapshot.get(repository=FS_REPO_NAME, snapshot='snap-single-index')['snapshots'][0]
console.print(f"  Indices in snapshot: {snap['indices']}")

──────────────────────────────────────────────── 2a. Single index ─────────────────────────────────────────────────

Indices in snapshot: ['kibana_sample_data_ecommerce']

In [4]:
heading('2b. Wildcard pattern — all sample data indices')
delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-wildcard')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='snap-wildcard',
    body={
        'indices': ['kibana_sample_data_*'],
        'include_global_state': False,
    },
    wait_for_completion=True,
)
snap = es.snapshot.get(repository=FS_REPO_NAME, snapshot='snap-wildcard')['snapshots'][0]
console.print(f"  Matched indices: {snap['indices']}")

───────────────────────────────── 2b. Wildcard pattern — all sample data indices ──────────────────────────────────

Matched indices: ['kibana_sample_data_ecommerce', '.ds-kibana_sample_data_logs-2026.04.06-000001', 
'kibana_sample_data_flights']

In [ ]:
heading('2c. Exclusion pattern — all sample data EXCEPT flights')
delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-exclude')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='snap-exclude',
    body={
        # The - prefix excludes; must come after a wildcard that would match it
        'indices': ['kibana_sample_data_*', '-kibana_sample_data_flights'],
        'include_global_state': False,
    },
    wait_for_completion=True,
)
snap = es.snapshot.get(repository=FS_REPO_NAME, snapshot='snap-exclude')['snapshots'][0]
console.print(f"  Indices (flights excluded): {snap['indices']}")

---
## 3. `ignore_unavailable` — Skip Missing Indices

In [5]:
heading('3. ignore_unavailable')

# Without ignore_unavailable — trying to snapshot a non-existent index fails
try:
    es.snapshot.create(
        repository=FS_REPO_NAME,
        snapshot='snap-missing-fail',
        body={
            'indices': ['kibana_sample_data_ecommerce', 'this-index-does-not-exist'],
            'ignore_unavailable': False,
        },
        wait_for_completion=True,
    )
except Exception as exc:
    warn(f'Expected failure: {exc}')

# With ignore_unavailable — the missing index is silently skipped
delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-ignore-unavailable')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='snap-ignore-unavailable',
    body={
        'indices': ['kibana_sample_data_ecommerce', 'this-index-does-not-exist'],
        'ignore_unavailable': True,
    },
    wait_for_completion=True,
)
snap = es.snapshot.get(repository=FS_REPO_NAME, snapshot='snap-ignore-unavailable')['snapshots'][0]
success(f"Succeeded with indices: {snap['indices']}")

────────────────────────────────────────────── 3. ignore_unavailable ──────────────────────────────────────────────

⚠ Expected failure: NotFoundError(404, 'index_not_found_exception', 'no such index ', this-index-does-not-exist, 
index_or_alias)

✓ Succeeded with indices: ['.ml-inference-native-000002', '.security-7', '.kibana_security_solution_9.3.0_001', 
'.secrets-inference', 'kibana_sample_data_ecommerce', '.apm-agent-configuration', '.ml-annotations-000001', 
'.kibana_usage_counters_9.3.0_001', '.kibana_analytics_9.3.0_001', '.kibana_task_manager_9.3.0_001', 
'.kibana_alerting_cases_9.3.0_001', '.kibana_9.3.0_001', '.kibana_locks-000001', '.integration_knowledge-7', 
'.security-profile-8', '.ml-inference-000005', '.kibana_ingest_9.3.0_001', '.kibana_search_solution_9.3.0_001', 
'.kibana_security_session_1', '.ml-notifications-000002', '.inference', '.apm-custom-link']

---
## 4. `include_global_state` — Capture Cluster State

In [6]:
heading('4. include_global_state: true vs false')

# Without global state — only index data
delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-no-global')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='snap-no-global',
    body={'indices': ['kibana_sample_data_ecommerce'], 'include_global_state': False},
    wait_for_completion=True,
)

# With global state — includes templates, pipelines, ILM, scripts
delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-with-global')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='snap-with-global',
    body={'indices': ['kibana_sample_data_ecommerce'], 'include_global_state': True},
    wait_for_completion=True,
)

for name in ['snap-no-global', 'snap-with-global']:
    s = es.snapshot.get(repository=FS_REPO_NAME, snapshot=name)['snapshots'][0]
    console.print(f"  {name}: include_global_state={s['include_global_state']}")

───────────────────────────────────── 4. include_global_state: true vs false ──────────────────────────────────────

snap-no-global: include_global_state=False

snap-with-global: include_global_state=True

---
## 5. `feature_states` — Capture Specific Feature Data

In [7]:
heading('5. Snapshot only the Kibana feature state (no indices)')

delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-kibana-only')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='snap-kibana-only',
    body={
        'indices': [],                   # no data indices
        'include_global_state': False,
        'feature_states': ['kibana'],    # only Kibana saved objects
    },
    wait_for_completion=True,
)
snap = es.snapshot.get(repository=FS_REPO_NAME, snapshot='snap-kibana-only')['snapshots'][0]
console.print(f"  Data indices    : {snap['indices']}")
console.print(f"  Feature states  : {[fs['feature_name'] for fs in snap.get('feature_states', [])]}")

───────────────────────────── 5. Snapshot only the Kibana feature state (no indices) ──────────────────────────────

Data indices    : ['.ds-kibana_sample_data_logs-2026.04.06-000001', '.kibana_usage_counters_9.3.0_001', 
'.slo-observability.sli-v3.6', '.internal.alerts-ml.anomaly-detection.alerts-default-000001', 
'.internal.alerts-streams.alerts-default-000001', '.ds-ilm-history-7-2026.04.06-000001', 
'.kibana_security_solution_9.3.0_001', '.kibana_9.3.0_001', '.kibana-siem-rule-migrations-prebuiltrules', 
'.ds-.edr-workflow-insights-default-2026.04.06-000001', '.kibana_analytics_9.3.0_001', 
'.ds-.logs-elasticsearch.deprecation-default-2026.04.06-000001', '.apm-agent-configuration', 
'.kibana_ingest_9.3.0_001', '.internal.alerts-observability.logs.alerts-default-000001', '.kibana_locks-000001', 
'.internal.alerts-security.attack.discovery.alerts-default-000001', '.slo-observability.summary-v3.6', 
'.internal.alerts-security.alerts-default-000001', '.internal.alerts-observability.apm.alerts-default-000001', 
'.internal.alerts-stack.alerts-default-000001', '.ds-.kibana-event-log-ds-2026.04.06-000001', 
'.kibana-siem-rule-migrations-integrations', '.apm-source-map', 
'.internal.alerts-observability.threshold.alerts-default-000001', '.ml-notifications-000002', 
'.slo-observability.summary-v3.6.temp', '.internal.alerts-observability.uptime.alerts-default-000001', 
'kibana_sample_data_flights', '.internal.alerts-observability.slo.alerts-default-000001', '.apm-custom-link', 
'.kibana_alerting_cases_9.3.0_001', '.internal.alerts-ml.anomaly-detection-health.alerts-default-000001', 
'.internal.alerts-default.alerts-default-000001', '.internal.alerts-dataset.quality.alerts-default-000001', 
'.kibana_search_solution_9.3.0_001', '.kibana_security_session_1', 'kibana_sample_data_ecommerce', 
'.internal.alerts-observability.metrics.alerts-default-000001', '.ml-annotations-000001', 
'.internal.alerts-transform.health.alerts-default-000001', '.kibana_task_manager_9.3.0_001']

Feature states  : ['kibana']

---
## 6. `partial` — Allow Partial Snapshots

Normally, if a shard is unavailable (e.g. node down), the snapshot fails.
`partial: true` allows the snapshot to proceed with whatever shards are available.

In [8]:
heading('6. partial: true')

# Create an index with a replica that can't be allocated (single-node cluster)
es.indices.create(
    index='training-unassigned',
    body={'settings': {'number_of_shards': 1, 'number_of_replicas': 2}},  # 2 replicas, 1 node = unassigned
    ignore=400,
)

health = es.cluster.health(index='training-unassigned', wait_for_status='yellow', timeout='10s')
console.print(f"  Index health    : {health['status']}")
console.print(f"  Unassigned shards: {health['unassigned_shards']}")

# partial=False would fail; partial=True succeeds
delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-partial')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='snap-partial',
    body={
        'indices': ['training-unassigned'],
        'include_global_state': False,
        'partial': True,
    },
    wait_for_completion=True,
)
snap = es.snapshot.get(repository=FS_REPO_NAME, snapshot='snap-partial')['snapshots'][0]
console.print(f"  Snapshot state   : {snap['state']}  (PARTIAL is expected here)")
console.print(f"  Failed shards    : {snap['shards']['failed']}")

es.indices.delete(index='training-unassigned', ignore_unavailable=True)

──────────────────────────────────────────────── 6. partial: true ─────────────────────────────────────────────────

/tmp/ipykernel_44739/6708274.py:4: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es.indices.create(


Index health    : yellow

Unassigned shards: 2

Snapshot state   : SUCCESS  (PARTIAL is expected here)

Failed shards    : 0

ObjectApiResponse({'acknowledged': True})

---
## 7. Snapshot Metadata

In [9]:
heading('7. Attaching metadata to a snapshot')

delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-with-metadata')
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='snap-with-metadata',
    body={
        'indices': ['kibana_sample_data_ecommerce'],
        'include_global_state': False,
        'metadata': {
            'taken_by': 'snapshot-training',
            'reason': 'pre-migration backup',
            'environment': 'training',
            'ticket': 'OPS-1234',
        },
    },
    wait_for_completion=True,
)
snap = es.snapshot.get(repository=FS_REPO_NAME, snapshot='snap-with-metadata')['snapshots'][0]
pp(snap['metadata'], 'Snapshot metadata')

─────────────────────────────────────── 7. Attaching metadata to a snapshot ───────────────────────────────────────

╭───────── Snapshot metadata ─────────╮
│ {                                   │
│   "taken_by": "snapshot-training",  │
│   "environment": "training",        │
│   "reason": "pre-migration backup", │
│   "ticket": "OPS-1234"              │
│ }                                   │
╰─────────────────────────────────────╯

---
## 8. Async Snapshots & Status Monitoring

In [ ]:
heading('8. Async snapshot — monitoring with _status')

delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-async')

# Start async (wait_for_completion=False returns immediately)
response = es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot='snap-async',
    body={'indices': ['kibana_sample_data_*'], 'include_global_state': False},
    wait_for_completion=False,
)
info(f'Snapshot started: {response}')

# Poll status
for _ in range(30):
    try:
        status = es.snapshot.status(repository=FS_REPO_NAME, snapshot='snap-async')
        snap_status = status['snapshots'][0] if status['snapshots'] else None
        if snap_status:
            state = snap_status['state']
            stats = snap_status.get('stats', {})
            console.print(f"  state={state}  files_total={stats.get('total', {}).get('file_count', '?')}  done={stats.get('processed', {}).get('file_count', '?')}")
            if state in ('SUCCESS', 'FAILED', 'PARTIAL'):
                break
    except Exception:
        pass
    time.sleep(1)

success('Async snapshot complete')

---
## 9. Snapshot Naming — Date Math

In [ ]:
heading('9. Date-math snapshot names')

# Date math uses URL encoding: < > / : are encoded
# Python client handles this automatically
import urllib.parse
from datetime import datetime

date_str = datetime.utcnow().strftime('%Y.%m.%d')
snap_name = f'snap-{date_str}'

delete_snapshot_if_exists(es, FS_REPO_NAME, snap_name)
es.snapshot.create(
    repository=FS_REPO_NAME,
    snapshot=snap_name,
    body={'indices': ['kibana_sample_data_ecommerce'], 'include_global_state': False},
    wait_for_completion=True,
)
success(f'Created snapshot: {snap_name}')

info('In SLM policies, use <snap-{now/d}> syntax for automatic date-stamped names.')

---
## 10. Cloning a Snapshot

In [ ]:
heading('10. Clone a snapshot')

# Clone creates an efficient server-side copy — no data is re-transferred through the client
delete_snapshot_if_exists(es, FS_REPO_NAME, 'snap-clone-target')

es.snapshot.clone(
    repository=FS_REPO_NAME,
    snapshot='snap-minimal',
    target_snapshot='snap-clone-target',
    body={'indices': 'kibana_sample_data_ecommerce'},  # can clone a subset of indices
)

snap = wait_for_snapshot(es, FS_REPO_NAME, 'snap-clone-target')
success(f"Clone state: {snap['state']}  indices: {snap['indices']}")

---
## 11. List All Snapshots in a Repository

In [ ]:
heading('11. All snapshots in the training repository')

all_snaps = es.snapshot.get(repository=FS_REPO_NAME, snapshot='*')
t = Table('Snapshot', 'State', 'Indices', 'Start time', 'Duration')
for s in all_snaps['snapshots']:
    duration_ms = s.get('duration_in_millis', 0)
    t.add_row(
        s['snapshot'],
        s['state'],
        str(len(s['indices'])),
        s.get('start_time', '—'),
        f'{duration_ms}ms',
    )
console.print(t)

## Summary

| Technique | Key parameter |
|-----------|---------------|
| Specific indices | `indices: ["my-index", "my-*"]` |
| Exclude an index | `indices: ["my-*", "-my-excluded"]` |
| Skip missing | `ignore_unavailable: true` |
| Include cluster state | `include_global_state: true` |
| Include Kibana objects | `feature_states: ["kibana"]` |
| Allow partial shards | `partial: true` |
| Attach notes | `metadata: {...}` |
| Server-side copy | `_clone` API |

**Next:** [04_saved_objects_deep_dive.ipynb](04_saved_objects_deep_dive.ipynb)